In [1]:
# import storage account helper
import sys
from pathlib import Path
from dotenv import load_dotenv
import os
project_root = Path().resolve().parent  # from neat_dashboard/ to repo root
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

ml_monolith_root = project_root / "ml_monolith"
if str(ml_monolith_root) not in sys.path:
    sys.path.insert(0, str(ml_monolith_root))

from ml_monolith.data_pipeline.storage_account_helpers import download_blob_to_dir, list_blobs_in_prefix
from ml_monolith.data_pipeline.process_raw_data import merge_json_files, label_data, label_data_v2, SensorRecording
load_dotenv()

storage_account_blob_uri = os.getenv("TRAINING_DATA_BLOB")
items_list = list_blobs_in_prefix(storage_account_blob_uri)

print("items in blob storage:")
for folder in items_list:
    print(folder)


2026-02-10 15:47:22,274 - ml_monolith.data_pipeline.storage_account_helpers - INFO - Using AZURE_STORAGE_CONNECTION_STRING for authentication.
2026-02-10 15:47:22,290 - ml_monolith.data_pipeline.storage_account_helpers - INFO - Listing blobs with prefix: 
2026-02-10 15:47:22,290 - azure.core.pipeline.policies.http_logging_policy - INFO - Request URL: 'https://harmlstorage.blob.core.windows.net/sensor-data?restype=REDACTED&comp=REDACTED&prefix=REDACTED'
Request method: 'GET'
Request headers:
    'x-ms-version': 'REDACTED'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.28.0 Python/3.10.0 (Windows-10-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '682a9639-068f-11f1-bb84-e884a56c3c92'
    'Authorization': 'REDACTED'
No body was attached to the request
2026-02-10 15:47:22,688 - azure.core.pipeline.policies.http_logging_policy - INFO - Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'a

items in blob storage:
08-02-2026/1770556157826.json
08-02-2026/1770556157826_seg0.json
08-02-2026/1770557152518.json
08-02-2026/1770557430117.json
08-02-2026/1770557595135.json
08-02-2026/1770561114570.json
08-02-2026/1770561114570_seg0.json
08-02-2026/merged_labels.csv
09-02-2026/1770621477170.json
09-02-2026/1770621477170_seg0.json
09-02-2026/1770624321078.json
09-02-2026/1770624340142.json
09-02-2026/1770624340142_seg0.json
09-02-2026/1770624340142_seg1.json
09-02-2026/1770624340142_seg10.json
09-02-2026/1770624340142_seg11.json
09-02-2026/1770624340142_seg12.json
09-02-2026/1770624340142_seg13.json
09-02-2026/1770624340142_seg14.json
09-02-2026/1770624340142_seg15.json
09-02-2026/1770624340142_seg16.json
09-02-2026/1770624340142_seg17.json
09-02-2026/1770624340142_seg18.json
09-02-2026/1770624340142_seg19.json
09-02-2026/1770624340142_seg2.json
09-02-2026/1770624340142_seg20.json
09-02-2026/1770624340142_seg21.json
09-02-2026/1770624340142_seg3.json
09-02-2026/1770624340142_seg4.j

In [2]:
download_dir = "neat_dashboard/downloaded_data"

download_blob_to_dir(storage_account_blob_uri, download_dir)

2026-02-10 15:47:26,480 - ml_monolith.data_pipeline.storage_account_helpers - INFO - Using AZURE_STORAGE_CONNECTION_STRING for authentication.
2026-02-10 15:47:26,482 - ml_monolith.data_pipeline.storage_account_helpers - INFO - Listing blobs with prefix: 
2026-02-10 15:47:26,485 - azure.core.pipeline.policies.http_logging_policy - INFO - Request URL: 'https://harmlstorage.blob.core.windows.net/sensor-data?restype=REDACTED&comp=REDACTED&prefix=REDACTED'
Request method: 'GET'
Request headers:
    'x-ms-version': 'REDACTED'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.28.0 Python/3.10.0 (Windows-10-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '6aaa9c92-068f-11f1-beac-e884a56c3c92'
    'Authorization': 'REDACTED'
No body was attached to the request
2026-02-10 15:47:26,807 - azure.core.pipeline.policies.http_logging_policy - INFO - Response status: 200
Response headers:
    'Transfer-Encoding': 'chunked'
    'Content-Type': 'a

In [3]:
folders = sorted(os.listdir(download_dir))

from datetime import datetime

def parse_date_prefix(name: str) -> datetime:
    date_part = name.split("_", 1)[0]
    return datetime.strptime(date_part, "%d-%m-%Y")

sorted_folders = sorted(folders, key=parse_date_prefix)
sorted_folders

['15-01-2026',
 '26-01-2026',
 '08-02-2026',
 '09-02-2026',
 '09-02-2026_2',
 '09-02-2026_3',
 '10-02-2026']

In [5]:
output_dir = "neat_dashboard/labeled_data"

for folder in sorted_folders:
    folder_to_process = os.path.join(download_dir, folder)
    merged_data = merge_json_files(folder_to_process, skipped_files=[])
    csv_files = [f for f in os.listdir(folder_to_process) if f.lower().endswith(".csv")]
    labels_csv_path = os.path.join(folder_to_process, csv_files[0]) if csv_files else None
    if folder in ['15-01-2026', '26-01-2026']:
        print(f"Folder {folder} needs to be processed with label_data.")
        labeled_data_df = label_data(merged_data, labels_csv_path)
    else:
        print(f"Folder {folder} needs to processed with label_data_v2.")
        labeled_data_df = label_data_v2(merged_data, labels_csv_path)
    
    output_path = os.path.join(output_dir, f"{folder}_labeled.csv")
    os.makedirs(output_dir, exist_ok=True)
    labeled_data_df.to_csv(output_path, index=False)


2026-02-10 15:50:22,724 - ml_monolith.data_pipeline.process_raw_data - INFO - Validating directory: neat_dashboard/downloaded_data\15-01-2026
2026-02-10 15:50:22,726 - ml_monolith.data_pipeline.process_raw_data - INFO - Directory neat_dashboard/downloaded_data\15-01-2026 exists.
2026-02-10 15:50:22,727 - ml_monolith.data_pipeline.process_raw_data - INFO - Path neat_dashboard/downloaded_data\15-01-2026 is a directory.
2026-02-10 15:50:22,729 - ml_monolith.data_pipeline.process_raw_data - INFO - Directory neat_dashboard/downloaded_data\15-01-2026 is not empty.
2026-02-10 15:50:23,680 - ml_monolith.data_pipeline.process_raw_data - ERROR - The file labels.csv is not a JSON file.
2026-02-10 15:50:23,680 - ml_monolith.data_pipeline.process_raw_data - INFO - File extension: .csv
2026-02-10 15:50:23,680 - ml_monolith.data_pipeline.process_raw_data - INFO - All files in directory neat_dashboard/downloaded_data\15-01-2026 are valid.
2026-02-10 15:50:23,680 - ml_monolith.data_pipeline.process_raw

Folder 15-01-2026 needs to be processed with label_data.


2026-02-10 15:50:24,237 - ml_monolith.data_pipeline.process_raw_data - INFO - Start Index for row with index 4 found: 2517
2026-02-10 15:50:24,237 - ml_monolith.data_pipeline.process_raw_data - INFO - Start Index for row with index 5 found: 3539
2026-02-10 15:50:24,237 - ml_monolith.data_pipeline.process_raw_data - INFO - Start Index for row with index 6 found: 3804
2026-02-10 15:50:24,237 - ml_monolith.data_pipeline.process_raw_data - INFO - Start Index for row with index 7 found: 5526
2026-02-10 15:50:24,237 - ml_monolith.data_pipeline.process_raw_data - INFO - Start Index for row with index 8 found: 6509
2026-02-10 15:50:24,247 - ml_monolith.data_pipeline.process_raw_data - INFO - Start Index for row with index 9 found: 6788
2026-02-10 15:50:24,250 - ml_monolith.data_pipeline.process_raw_data - INFO - Start Index for row with index 10 found: 7094
2026-02-10 15:50:24,253 - ml_monolith.data_pipeline.process_raw_data - INFO - Start Index for row with index 11 found: 11356
2026-02-10 15

Folder 26-01-2026 needs to be processed with label_data.


2026-02-10 15:50:25,607 - ml_monolith.data_pipeline.process_raw_data - INFO - Validating directory: neat_dashboard/downloaded_data\08-02-2026
2026-02-10 15:50:25,609 - ml_monolith.data_pipeline.process_raw_data - INFO - Directory neat_dashboard/downloaded_data\08-02-2026 exists.
2026-02-10 15:50:25,609 - ml_monolith.data_pipeline.process_raw_data - INFO - Path neat_dashboard/downloaded_data\08-02-2026 is a directory.
2026-02-10 15:50:25,609 - ml_monolith.data_pipeline.process_raw_data - INFO - Directory neat_dashboard/downloaded_data\08-02-2026 is not empty.
2026-02-10 15:50:25,696 - ml_monolith.data_pipeline.process_raw_data - ERROR - The file merged_labels.csv is not a JSON file.
2026-02-10 15:50:25,696 - ml_monolith.data_pipeline.process_raw_data - INFO - File extension: .csv
2026-02-10 15:50:25,696 - ml_monolith.data_pipeline.process_raw_data - INFO - All files in directory neat_dashboard/downloaded_data\08-02-2026 are valid.
2026-02-10 15:50:25,696 - ml_monolith.data_pipeline.proc

...

Folder 09-02-2026 needs to processed with label_data_v2.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58619 entries, 0 to 58618
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   accelerometerX  58619 non-null  float64
 1   accelerometerY  58619 non-null  float64
 2   accelerometerZ  58619 non-null  float64
 3   gyroscopeX      58619 non-null  float64
 4   gyroscopeY      58619 non-null  float64
 5   gyroscopeZ      58619 non-null  float64
 6   timestamp       58619 non-null  int64  
 7   timestampNanos  58619 non-null  int64  
 8   label           2117 non-null   object 
dtypes: float64(6), int64(2), object(1)
memory usage: 4.0+ MB
Folder 09-02-2026_2 needs to processed with label_data_v2.


2026-02-10 15:50:26,952 - ml_monolith.data_pipeline.process_raw_data - INFO - Processing 2 labels...
2026-02-10 15:50:26,964 - ml_monolith.data_pipeline.process_raw_data - INFO - Label 1: Stairs Down
2026-02-10 15:50:26,965 - ml_monolith.data_pipeline.process_raw_data - INFO -   Original range: 1770644678745 to 1770644723179
2026-02-10 15:50:26,965 - ml_monolith.data_pipeline.process_raw_data - INFO -   Buffered range: 1770644678745 to 1770644723179
2026-02-10 15:50:26,967 - ml_monolith.data_pipeline.process_raw_data - INFO - Labeled 13 sensor data points
2026-02-10 15:50:26,969 - ml_monolith.data_pipeline.process_raw_data - INFO - Label 2: Stairs Up
2026-02-10 15:50:26,969 - ml_monolith.data_pipeline.process_raw_data - INFO -   Original range: 1770644736599 to 1770644783209
2026-02-10 15:50:26,969 - ml_monolith.data_pipeline.process_raw_data - INFO -   Buffered range: 1770644736599 to 1770644783209
2026-02-10 15:50:26,969 - ml_monolith.data_pipeline.process_raw_data - INFO - Labeled 2

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 478 entries, 0 to 477
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   accelerometerX  478 non-null    float64
 1   accelerometerY  478 non-null    float64
 2   accelerometerZ  478 non-null    float64
 3   gyroscopeX      478 non-null    float64
 4   gyroscopeY      478 non-null    float64
 5   gyroscopeZ      478 non-null    float64
 6   timestamp       478 non-null    int64  
 7   timestampNanos  478 non-null    int64  
 8   label           278 non-null    object 
dtypes: float64(6), int64(2), object(1)
memory usage: 33.7+ KB
Folder 09-02-2026_3 needs to processed with label_data_v2.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3018 entries, 0 to 3017
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   accelerometerX  3018 non-null   float64
 1   accelerometerY  3018 non-null   f

2026-02-10 15:50:27,165 - ml_monolith.data_pipeline.process_raw_data - INFO - Skipping file: labelss.csv
2026-02-10 15:50:27,168 - ml_monolith.data_pipeline.process_raw_data - INFO - Total sensor data points: 3414
2026-02-10 15:50:27,179 - ml_monolith.data_pipeline.process_raw_data - INFO - Timestamp range: 1770662751600 to 1770667757996
2026-02-10 15:50:27,180 - ml_monolith.data_pipeline.process_raw_data - INFO - Processing 3 labels...
2026-02-10 15:50:27,181 - ml_monolith.data_pipeline.process_raw_data - INFO - Label 1: Stairs Down
2026-02-10 15:50:27,182 - ml_monolith.data_pipeline.process_raw_data - INFO -   Original range: 1770662854047 to 1770662889301
2026-02-10 15:50:27,182 - ml_monolith.data_pipeline.process_raw_data - INFO -   Buffered range: 1770662854047 to 1770662889301
2026-02-10 15:50:27,186 - ml_monolith.data_pipeline.process_raw_data - INFO - Labeled 218 sensor data points
2026-02-10 15:50:27,186 - ml_monolith.data_pipeline.process_raw_data - INFO - Label 2: Stairs Up


Folder 10-02-2026 needs to processed with label_data_v2.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3414 entries, 0 to 3413
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   accelerometerX  3414 non-null   float64
 1   accelerometerY  3414 non-null   float64
 2   accelerometerZ  3414 non-null   float64
 3   gyroscopeX      3414 non-null   float64
 4   gyroscopeY      3414 non-null   float64
 5   gyroscopeZ      3414 non-null   float64
 6   timestamp       3414 non-null   int64  
 7   timestampNanos  3414 non-null   int64  
 8   label           1909 non-null   object 
dtypes: float64(6), int64(2), object(1)
memory usage: 240.2+ KB


In [6]:
import pandas as pd
generation_seed_dir = "neat_dashboard/generation_seed"
os.makedirs(generation_seed_dir, exist_ok=True)


def merge_csv_files(csv_dir):
    merged_df = pd.DataFrame()
    files_sorted = sorted(os.listdir(csv_dir))
    for filename in files_sorted:
        if filename.endswith('.csv'):
            df = pd.read_csv(os.path.join(csv_dir, filename))
            merged_df = pd.concat([merged_df, df], ignore_index=True)
    return merged_df


merged_csv_df = merge_csv_files(output_dir)
merged_csv_path = os.path.join(generation_seed_dir, "merged_labeled_data.csv")
merged_csv_df.to_csv(merged_csv_path, index=False)

